In [171]:
import os
import pickle
import sys
import pandas as pd
import random
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib import cm

In [172]:
region = "13"
path = f"../data/pkl/{region}/"
if os.path.exists(path):
    files = os.listdir(path)
    print(files)
else:
    print(f"Path {path} does not exist.")
    exit()

['13_max_time', '13_lp_kommuner']


In [173]:
with open(path + f"{region}_max_time", "rb") as f:
    time = pickle.load(f)

with open(path + f"{region}_lp_kommuner", "rb") as f:
    lp = pickle.load(f)

In [174]:
for key, values in time.items():
    print(f"{key}: {values}")

Varberg: {'EF_2030_RAPS_24_V1.parquet': [np.int64(1832), np.float64(0.2546265856618917), array([0.02882894, 0.02874086, 0.02843204, ..., 0.02022857, 0.020039  ,
       0.01982151], shape=(8760,))], 'EF_2022_RAPS_27_V1.parquet': [np.int64(8047), np.float64(10.128938248289726), array([0.70305786, 0.70305786, 0.71123295, ..., 0.17167692, 0.16350183,
       0.17167692], shape=(8760,))], 'EF_2030_RAPS_22_V1.parquet': [np.int64(0), np.float64(0.0), array([0., 0., 0., ..., 0., 0., 0.], shape=(8760,))], 'EF_2030_RAPS_7_V1.parquet': [np.int64(4257), np.float64(12.855695896711417), array([5.37588417, 5.3975318 , 5.34556895, ..., 5.22564106, 5.28718863,
       5.37304495], shape=(8760,))], 'EF_2027_RAPS_22_V1.parquet': [np.int64(0), np.float64(0.0), array([0., 0., 0., ..., 0., 0., 0.], shape=(8760,))], 'EF_2040_RAPS_17_V1.parquet': [np.int64(0), np.float64(0.0), array([0., 0., 0., ..., 0., 0., 0.], shape=(8784,))], 'EF_2022_RAPS_23_V1.parquet': [np.int64(1018), np.float64(1.381738808897012), arra

In [175]:
for key, values in lp.items():
    print(key)
    print(values)
    print("")

Varberg
{'2022': array([ 95.01559682,  92.29986986,  90.42326763, ..., 117.23802689,
       110.15762849, 102.81106081], shape=(8760,)), '2027': array([ 97.7469894 ,  94.75486316,  92.82654263, ..., 122.34628816,
       114.22841799, 106.29050158], shape=(8760,)), '2030': array([100.06902328,  96.82355163,  94.8288215 , ..., 126.8733369 ,
       117.79068914, 109.27604743], shape=(8760,)), '2040': array([131.2915685 , 127.36958468, 125.26992635, ..., 149.8309712 ,
       141.18545031, 140.1578555 ], shape=(8784,))}

Falkenberg
{'2022': array([65.14054738, 62.5754803 , 60.89061487, ..., 87.64946932,
       81.56333837, 75.02950855], shape=(8760,)), '2027': array([66.25651658, 63.51532834, 61.81899533, ..., 90.21624744,
       83.56174178, 76.70371029], shape=(8760,)), '2030': array([69.59864489, 66.68451761, 64.97100129, ..., 95.11406612,
       87.8584617 , 80.64700354], shape=(8760,)), '2040': array([76.50270884, 73.12384503, 71.37040608, ..., 99.19408909,
       91.74400239, 91.17441

In [176]:
def get_time_data(time_dict, kommun, year):
    filtered_times = {" ".join(k.split("_")[2:-1]): [int(v[0]), float(v[1]), v[2]] for k, v in time_dict[kommun].items() if year in k}
    return filtered_times

def get_max_and_index(loadprofile):
    max_value = np.max(loadprofile)
    index = np.argmax(loadprofile)
    return max_value, index

# Your custom key order
custom_order = [
    "Flerbostadshus", "Smahus", "LOM Flerbostadshus", "LOM Smahus", "RAPS 1 3", "RAPS 4", "RAPS 5", "RAPS 6", "RAPS 7", "RAPS 8", "RAPS 9", "RAPS 10", "RAPS 11", "RAPS 12", "RAPS 13", "RAPS 14", "RAPS 15",
    "RAPS 16", "RAPS 17", "RAPS 18", "RAPS 19", "RAPS 20", "RAPS 21", "RAPS 22", "RAPS 23", "RAPS 24", "RAPS 27", "RAPS 7777", "RAPS 8888", "PB", "LL", "TT DEP", "TT DEST", "TT RESTSTOP",
]

for kommun, _ in lp.items():
    rows = {}
    df_year = pd.DataFrame()
    c1 = []
    c2 = []
    c3 = []
    for year, loadprofile in _.items():
        if year not in rows:
            rows[year] = []
        xrange = range(len(loadprofile))
        filtered_times = get_time_data(time, kommun, year)
        max_value, index = get_max_and_index(loadprofile)
        # rows.append({"År": year, "MaxTidpunkt": index, "MaxTidpunktEB": max_value})
        d1 = []
        d2 = []
        c1.append(max_value)
        c2.append(index)
        c3.append(year)
        for k, v in filtered_times.items():

            d1.append(k)
            d2.append(v[2][index])
        rows[year] = d1
        df_tmp = pd.DataFrame(d2, index=d1, columns=[year])
  
        if df_year.empty:
            df_year = df_tmp
        else:
            df_year = pd.merge(df_year, df_tmp, left_index=True, right_index=True, how="outer")

    df_year_max = pd.DataFrame({"Tidpunkt": c2, "Max": c1}, index=c3).T

    df_year = df_year.reindex(custom_order)
    df_year.replace(0, np.nan, inplace=True)

    # Save to the same sheet
    out_path = f"../data/excel/{region}/"
    if not os.path.exists(out_path):
        os.makedirs(out_path)
    with pd.ExcelWriter(os.path.join(out_path, f"{kommun} - Maxeffekt.xlsx"), engine="xlsxwriter") as writer:
        df_year_max.to_excel(writer, sheet_name="Combined")
        
        # Leave 2 empty rows between the tables for clarity
        start_row = len(df_year_max) + 3
        
        df_year.to_excel(writer, sheet_name="Combined", startrow=start_row)